# Exploratory raw-waveform cross-correlation
This notebook reviews **waveform similarity**, not physical source. The primary search uses separate stereo channels, local zero-mean normalization, and the fixed 0.30-second reference crop. It uses no filtering or denoising. A high score does not mean the same sound or a germination sound. The reference self-match is excluded from external rankings.


In [ ]:
from pathlib import Path
import sys
PROJECT=Path.cwd().resolve()
if PROJECT.name=='notebooks': PROJECT=PROJECT.parent
if not (PROJECT/'src/audio_cross_correlation.py').exists(): raise RuntimeError('Open inside seed-germination')
sys.path.insert(0,str(PROJECT))
import pandas as pd
import ipywidgets as widgets
from IPython.display import Audio,Image,Markdown,clear_output,display
TABLES=PROJECT/'results/tables/targeted_multimodal'
FIGURES=PROJECT/'results/figures/targeted_multimodal/cross_correlation'
TOP=pd.read_csv(TABLES/'cross_correlation_top_matches.csv')
SUMMARY=pd.read_csv(TABLES/'cross_correlation_summary.csv')
MANUAL=TABLES/'cross_correlation_manual_review.csv'
ALIGNMENT=pd.read_csv(TABLES/'cross_correlation_visual_alignment.csv')
SENSITIVITY=pd.read_csv(TABLES/'cross_correlation_template_sensitivity.csv')
print('Primary template:',SUMMARY.loc[0,'reference_crop_start_seconds'],'to',SUMMARY.loc[0,'reference_crop_end_seconds'],'seconds')
display(SUMMARY)
display(TOP[['rank','filename','absolute_match_timestamp','max_zncc','lag_seconds','channel','local_candidate_RMS']].head(50))


## Reference and full-corpus score distribution
The plots use the original waveform. Near-zero-energy comparison windows are skipped for numerical safety. The time axis in the top-match timeline spans the full experiment; WAV gaps do not imply silence.


In [ ]:
for name in ['reference_transient_waveform.png','reference_transient_spectrogram.png','correlation_score_distribution.png','correlation_score_histogram_zoomed_top_tail.png','top_matches_timeline.png','top_matches_by_day.png','template_rank_stability.png']:
    print(name);display(Image(filename=FIGURES/name))


## Listen to a top non-reference match
The review copies are unchanged originals. Use the aligned waveform and spectrogram figure to judge whether an apparent match is acoustically meaningful. Enter neutral human descriptions in `cross_correlation_manual_review.csv`; the table begins blank. Do not call a clip a germination sound.


In [ ]:
review=pd.read_csv(MANUAL,keep_default_na=False)
choice=widgets.Dropdown(options=[(f'{r.rank:02d} | {r.filename} | r={r.max_zncc:.4f}',int(r.rank)) for r in review.itertuples(index=False)],description='Match:',layout=widgets.Layout(width='850px'))
out=widgets.Output()
def show(*_):
    row=review.loc[review['rank'].eq(choice.value)].iloc[0]
    number=choice.value; filename=row.filename; score=row.max_zncc; timestamp=row.timestamp; rms=row.local_RMS
    with out:
        clear_output(wait=True)
        display(Markdown(f'### Rank {number}: {filename}\nZNCC={score:.4f}; matched peak time {timestamp}; local RMS={rms:.6g}'))
        display(Image(filename=FIGURES/f'top_match_{choice.value:02d}.png'))
        display(Audio(filename=row['review_copy_path'],normalize=False))
choice.observe(show,names='value');display(choice,out);show()


## Descriptive visual alignment and template sensitivity
Visual event times did not affect correlation scores or ranking. Alternative 0.20- and 0.40-second templates were evaluated only within the primary top-200 pool, so their ranks are **pool-limited**, not full-corpus ranks.


In [ ]:
display(ALIGNMENT)
display(SENSITIVITY.head(50))
print('Manual review table:',MANUAL)
